In [0]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timezone, timedelta

dbutils.widgets.dropdown("mode", "manual", ["auto", "manual"], "Mode")
dbutils.widgets.text("start_date", "2026-10-01", "Manual Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "2026-10-01", "Manual End Date (YYYY-MM-DD)")

MODE = dbutils.widgets.get("mode")
MANUAL_START = dbutils.widgets.get("start_date").strip()
MANUAL_END = dbutils.widgets.get("end_date").strip()

CATALOG = "workspace"
DB = f"{CATALOG}.complaints"
WATERMARK = f"{DB}.pipeline_watermark"

OUTPUT_DIR = "/Workspace/Users/furyboy4592@gmail.com/Consumer-Financial-Complaint-Analytics/data/samples"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUT_FILE = os.path.join(OUTPUT_DIR, "incremental_load_sample.csv")

BASE_URL = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
HEADERS = {
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}
PAGE_SIZE = 1000  # API supports up to 1000 per page with search_after

def get_date_window():
    if MODE == "manual" and MANUAL_START and MANUAL_END:
        return (datetime.strptime(MANUAL_START, "%Y-%m-%d").date(),
                datetime.strptime(MANUAL_END, "%Y-%m-%d").date())
    
    # Auto mode: lag by 2 days to account for CFPB processing delay
    target_date = datetime.now(timezone.utc).date() - timedelta(days=2)
    try:
        if spark.catalog.tableExists(WATERMARK):
            wm = spark.table(WATERMARK).filter("layer = 'silver'").select("last_processed_date").collect()
            if wm and wm[0][0]:
                start = wm[0][0] + timedelta(days=1)
                return (start, target_date)
    except Exception:
        pass
    return (target_date, target_date)

def fetch_search_after_page(date_str, search_after_token=None, retries=3):
    params = {
        "size": PAGE_SIZE,
        "date_received_min": date_str,
        "date_received_max": date_str,
        "sort": "created_date_desc"
    }
    if search_after_token:
        params["search_after"] = search_after_token

    for attempt in range(retries):
        try:
            res = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=45)
            if res.status_code == 403:
                print("Access denied / rate limited. Sleeping 5 seconds...")
                time.sleep(5)
                continue
            res.raise_for_status()
            return res.json().get("hits", {}).get("hits", [])
        except requests.exceptions.RequestException as e:
            time.sleep(2)
    return []

def extract_day_data(day_date):
    day_str = day_date.isoformat()
    all_rows = []
    search_after_token = None
    page_num = 1

    print(f"\n--- Extracting {day_str} via search_after cursor ---")
    while True:
        hits = fetch_search_after_page(day_str, search_after_token)
        if not hits:
            break

        for h in hits:
            all_rows.append(h["_source"])

        # Stop when fewer records than page size return
        if len(hits) < PAGE_SIZE:
            break

        # Build next search_after token from last hit: e.g. "1790899188000_28830795"
        last_sort = hits[-1].get("sort")
        if not last_sort or len(last_sort) < 2:
            break
        search_after_token = f"{last_sort[0]}_{last_sort[1]}"
        
        if page_num % 5 == 0:
            print(f"  Page {page_num}: collected {len(all_rows)} complaints so far...")

        page_num += 1
        time.sleep(1.0)  # Rate limiting per TA instructions

    print(f"Completed {day_str}: {len(all_rows)} total complaints retrieved.")
    return all_rows

start_dt, end_dt = get_date_window()
if start_dt > end_dt:
    print(f"Up to date through {end_dt}.")
    dbutils.notebook.exit("UP_TO_DATE")

all_records = []
curr = start_dt
while curr <= end_dt:
    all_records.extend(extract_day_data(curr))
    curr += timedelta(days=1)

if not all_records:
    print("No records retrieved.")
    dbutils.notebook.exit("NO_DATA")

df_extracted = pd.DataFrame(all_records)
df_extracted.to_csv(OUT_FILE, index=False)
print(f"\nSaved {len(df_extracted)} rows to {OUT_FILE}.")
dbutils.notebook.exit("SUCCESS")

In [0]:
import pandas as pd

path = "/Workspace/Users/furyboy4592@gmail.com/Consumer-Financial-Complaint-Analytics/data/samples/incremental_load_sample.csv"
df = pd.read_csv(path)
df_clean = df.drop_duplicates(subset=["complaint_id"])
df_clean.to_csv(path, index=False)
print(f"Cleaned CSV: {len(df)} rows collapsed to {len(df_clean)} unique complaints.")